# Introduction to Machine Learning with scikit-learn
---

**Copyright and License**

© 2026, Fiorella Piriz Sapio

This Jupyter Notebook is licensed under the [MIT License](https://opensource.org/licenses/MIT).

**Disclaimer:**  
This notebook is provided "as is", without warranty of any kind, express or implied.  
The author assumes no responsibility or liability for any errors, omissions, or outcomes resulting from the use of this notebook or its contents.  
All analyses and interpretations are for educational and research purposes only and do not constitute medical or clinical advice.

**Dataset Note:**  
This notebook uses built-in scikit-learn datasets for demonstration purposes. These are well-known benchmark datasets widely used in the machine learning community.

## Abstract: Introduction to scikit-learn

[scikit-learn](https://scikit-learn.org/) is the most widely used Python library for classical machine learning. It provides a consistent, clean API for building, training, and evaluating models — from simple linear regression to ensemble methods.

Built on top of NumPy, SciPy, and Matplotlib, scikit-learn integrates naturally into the scientific Python ecosystem.

### Why scikit-learn?
- **Unified API:** Every model follows the same `fit` / `predict` / `score` interface, making it easy to swap algorithms.
- **Breadth:** Covers supervised learning, unsupervised learning, preprocessing, model selection, and evaluation.
- **Production-ready:** Models can be serialized and deployed with minimal overhead.
- **Ecosystem:** Integrates seamlessly with pandas, NumPy, and visualization libraries.

### Learning Objectives
By completing this notebook, students will be able to:
- Understand the scikit-learn estimator API (`fit`, `predict`, `score`)
- Split data into training and testing sets with `train_test_split`
- Train and evaluate a **Linear Regression** model
- Train and evaluate a **Logistic Regression** classifier
- Interpret key metrics: MSE, R², accuracy, confusion matrix
- Visualize model performance with a **ROC curve**
- Preprocess features with `StandardScaler`
- Evaluate models robustly using **cross-validation**

### Real-World Context
scikit-learn is used across virtually every data-driven domain:
- **Telecom:** Predicting churn, anomaly detection in network KPIs.
- **Healthcare:** Disease classification, patient risk scoring.
- **Finance:** Credit scoring, fraud detection.
- **Engineering:** Predictive maintenance, quality control.

## 1. Setup & Library Imports

We import scikit-learn modules alongside NumPy and Matplotlib. Each import will be explained in context as we use it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Data splitting
from sklearn.model_selection import train_test_split

# Models
from sklearn.linear_model import LinearRegression, LogisticRegression

# Metrics
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    accuracy_score,
    confusion_matrix,
    roc_curve,
    auc
)

# Built-in datasets
from sklearn.datasets import load_diabetes, load_breast_cancer

## 2. The scikit-learn API

Every estimator in scikit-learn follows the same three-step pattern:

| Step | Method | Description |
|------|--------|-------------|
| 1 | `model = Algorithm()` | Instantiate the model with hyperparameters |
| 2 | `model.fit(X_train, y_train)` | Learn from training data |
| 3 | `model.predict(X_test)` | Generate predictions on new data |

This consistency means switching from `LinearRegression` to `RandomForestRegressor` requires changing only one line.

## 3. Splitting Data: train_test_split

Before training any model, we split the dataset into:
- **Training set:** used to fit the model.
- **Test set:** held out to evaluate generalization on unseen data.

Key parameters:
- `test_size`: fraction of data reserved for testing (e.g. `0.2` = 20%).
- `random_state`: seed for reproducibility.

In [ ]:
# Simple example with synthetic data
X = np.array([[1], [2], [3], [4], [5], [6], [7], [8], [9], [10]])
y = np.array([2, 4, 5, 4, 5, 7, 8, 9, 10, 12])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Total samples:", len(X))
print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

## 4. Supervised Learning — Regression

**Regression** predicts a continuous numerical output.

**Linear Regression** fits a line (or hyperplane) that minimizes the sum of squared residuals:

$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \ldots$$

We'll use the **Diabetes dataset** — 442 patients, 10 physiological features, target is a quantitative measure of disease progression.

In [ ]:
# Load dataset
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target

print("Features:", diabetes.feature_names)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("y range: [{:.1f}, {:.1f}]".format(y.min(), y.max()))

In [ ]:
# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Train
reg = LinearRegression()
reg.fit(X_train, y_train)

In [ ]:
# Predict
y_pred = reg.predict(X_test)

print("First 5 predictions:", y_pred[:5].round(1))
print("First 5 actual values:", y_test[:5])

### 4.1 Regression Metrics

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **MSE** | mean of squared errors | Lower is better; same units² as target |
| **RMSE** | √MSE | Same units as target, easier to interpret |
| **R²** | 1 - SS_res/SS_tot | 1.0 = perfect fit; 0 = predicts mean only |

In [ ]:
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print(f"MSE:  {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.4f}")

In [ ]:
# Visualize predictions vs actual
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred, alpha=0.6, color='steelblue', label='Predictions')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         color='red', linestyle='--', label='Perfect fit')
plt.xlabel("Actual values")
plt.ylabel("Predicted values")
plt.title("Linear Regression: Predicted vs Actual")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Inspect learned coefficients
print("Intercept:", reg.intercept_.round(2))
for name, coef in zip(diabetes.feature_names, reg.coef_):
    print(f"  {name}: {coef:.2f}")

## 5. Supervised Learning — Classification

**Classification** predicts a discrete class label.

**Logistic Regression** models the probability that a sample belongs to a class using the sigmoid function:

$$P(y=1 \mid x) = \frac{1}{1 + e^{-(w^T x + b)}}$$

We'll use the **Breast Cancer dataset** — 569 samples, 30 features, binary target: malignant (0) or benign (1).

In [ ]:
# Load dataset
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target

print("Classes:", cancer.target_names)
print("X shape:", X.shape)
print("Class distribution — 0 (malignant):", (y == 0).sum(),
      "| 1 (benign):", (y == 1).sum())

In [ ]:
# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Train
clf = LogisticRegression(max_iter=10000, random_state=42)
clf.fit(X_train, y_train)

In [ ]:
# Predict class labels
y_pred = clf.predict(X_test)

# Predict probabilities for the positive class (benign = 1)
y_prob = clf.predict_proba(X_test)[:, 1]

print("First 5 predictions:", y_pred[:5])
print("First 5 actual labels:", y_test[:5])
print("First 5 probabilities (benign):", y_prob[:5].round(3))

### 5.1 Classification Metrics

| Metric | Description |
|--------|-------------|
| **Accuracy** | Fraction of correct predictions |
| **Confusion Matrix** | Table of TP, TN, FP, FN counts |
| **ROC Curve** | Trade-off between True Positive Rate and False Positive Rate |
| **AUC** | Area under the ROC curve; 1.0 = perfect, 0.5 = random |

In [ ]:
acc = accuracy_score(y_test, y_pred)
cm  = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print("\nConfusion Matrix:")
print(cm)
print("\n  Rows = Actual | Columns = Predicted")
print(f"  TN={cm[0,0]}  FP={cm[0,1]}")
print(f"  FN={cm[1,0]}  TP={cm[1,1]}")

### 5.2 ROC Curve

The **ROC (Receiver Operating Characteristic) curve** plots the True Positive Rate (TPR) against the False Positive Rate (FPR) at every classification threshold.

- A curve hugging the top-left corner indicates a strong classifier.
- The diagonal dashed line represents a random classifier (AUC = 0.5).
- **AUC** summarizes the curve in a single number — higher is better.

In [ ]:
# Compute ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

print(f"AUC: {roc_auc:.4f}")

In [ ]:
# Plot ROC curve
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='steelblue', lw=2,
         label=f'Logistic Regression (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Random classifier')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Breast Cancer Classification")
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 6. Preprocessing — Feature Scaling

Most ML algorithms are sensitive to the scale of input features. For example, if one feature ranges from 0–1 and another from 0–100,000, the model may incorrectly treat the larger-valued feature as more important.

**`StandardScaler`** standardizes features by removing the mean and scaling to unit variance:

$$x' = \frac{x - \mu}{\sigma}$$

After scaling, each feature has **mean = 0** and **standard deviation = 1**.

### The golden rule of preprocessing

Always fit the scaler **only on training data**, then apply it to both train and test sets. Fitting on the full dataset would leak information about the test set into the model — a form of **data leakage**.

| Step | Training set | Test set |
|------|-------------|----------|
| Fit scaler | ✅ `fit_transform(X_train)` | ❌ never fit here |
| Apply scaler | ✅ included above | ✅ `transform(X_test)` |

In [ ]:
from sklearn.preprocessing import StandardScaler

# Reload breast cancer data and split
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit ONLY on training data, then transform both sets
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform
X_test_scaled  = scaler.transform(X_test)        # transform only

print("Before scaling — X_train mean:", X_train[:, 0].mean().round(4),
      "| std:", X_train[:, 0].std().round(4))
print("After scaling  — X_train mean:", X_train_scaled[:, 0].mean().round(4),
      "| std:", X_train_scaled[:, 0].std().round(4))

In [ ]:
# Compare LogisticRegression with and without scaling
clf_raw    = LogisticRegression(max_iter=10000, random_state=42)
clf_scaled = LogisticRegression(max_iter=10000, random_state=42)

clf_raw.fit(X_train, y_train)
clf_scaled.fit(X_train_scaled, y_train)

acc_raw    = accuracy_score(y_test, clf_raw.predict(X_test))
acc_scaled = accuracy_score(y_test, clf_scaled.predict(X_test_scaled))

print(f"Accuracy without scaling: {acc_raw:.4f}")
print(f"Accuracy with scaling:    {acc_scaled:.4f}")

## 7. Model Evaluation — Cross-Validation

A single train/test split can give a misleading score depending on which samples end up in each set. **K-Fold Cross-Validation** solves this by:

1. Splitting the data into **k equal folds**.
2. Training on k-1 folds and evaluating on the remaining fold.
3. Repeating k times so every sample is used for testing exactly once.
4. Reporting the **mean ± std** of the k scores.

```
Fold 1: [TEST] [train] [train] [train] [train]
Fold 2: [train] [TEST] [train] [train] [train]
Fold 3: [train] [train] [TEST] [train] [train]
Fold 4: [train] [train] [train] [TEST] [train]
Fold 5: [train] [train] [train] [train] [TEST]
```

This gives a much more **reliable estimate** of how the model will perform on unseen data, and also reveals if the model is unstable (high std across folds).

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Use a Pipeline to avoid data leakage during cross-validation
# Pipeline applies scaling inside each fold automatically
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=10000, random_state=42))
])

X, y = cancer.data, cancer.target

scores = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')

print("Accuracy per fold:", scores.round(4))
print(f"Mean: {scores.mean():.4f} | Std: {scores.std():.4f}")

In [ ]:
# Visualize fold scores
plt.figure(figsize=(6, 4))
plt.bar(range(1, 6), scores, color='steelblue', alpha=0.8)
plt.axhline(scores.mean(), color='red', linestyle='--',
            label=f'Mean = {scores.mean():.4f}')
plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.title("5-Fold Cross-Validation Scores")
plt.ylim(0.9, 1.0)
plt.legend()
plt.tight_layout()
plt.show()

### Why use a Pipeline?

Notice we wrapped the scaler and classifier in a `Pipeline`. This is critical when using cross-validation.

Without a pipeline, if you scale the full dataset before calling `cross_val_score`, the scaler has already seen the test fold — **data leakage**. The pipeline ensures the scaler is fit only on the training folds in each iteration, keeping the evaluation honest.

You'll use pipelines extensively in the upcoming labs.